# Exercise 48 — `@dataclass`

## Concept

For classes that mostly hold data, writing `__init__`, `__repr__`, and `__eq__` by hand is repetitive. `@dataclass` generates them for you from the type-annotated class attributes:

```python
from dataclasses import dataclass

@dataclass
class User:
    id: int
    name: str
    email: str

u = User(1, "Alice", "a@x.com")
repr(u)              # "User(id=1, name='Alice', email='a@x.com')"
u == User(1, "Alice", "a@x.com")    # True — field-by-field equality, also auto-generated
```

### Defaults

Give a default value just like a normal parameter:

```python
@dataclass
class User:
    id: int
    name: str
    active: bool = True
```

For **mutable defaults** (list, dict, set), you must use `field(default_factory=...)` — same reasoning as ex_32's mutable-default trap:

```python
from dataclasses import field

@dataclass
class Batch:
    items: list = field(default_factory=list)
```

### `frozen=True` for immutability

Makes instances immutable (attempts to set attributes raise `FrozenInstanceError`) and makes the class hashable — usable as dict keys / set elements.

```python
@dataclass(frozen=True)
class Coord:
    x: int
    y: int
```

When to use a dataclass vs a plain class: **default to dataclass** for data containers. Reach for a plain class when you have lots of behavior (many methods) or unusual construction logic.

## Task 1 — Replace a hand-written class

Rebuild the `Point` class from ex_46 as a `@dataclass` with `x: int` and `y: int`. Auto-generated `__init__`, `__repr__`, and `__eq__` should already give you everything the assertions need.

Expected:
- `Point(3, 4).x == 3`, `Point(3, 4).y == 4`
- `repr(Point(3, 4))` → `"Point(x=3, y=4)"`
- `Point(3, 4) == Point(3, 4)` (without writing `__eq__`)

In [ ]:
from dataclasses import dataclass

# TODO: define Point as a @dataclass

p = Point(3, 4)
assert p.x == 3 and p.y == 4
assert repr(p) == "Point(x=3, y=4)"
assert Point(3, 4) == Point(3, 4)
assert Point(3, 4) != Point(3, 5)
print("ok")


## Task 2 — Defaults, including a mutable one

Define a `Batch` dataclass with:
- `name: str` (required)
- `items: list[int]` (defaults to an empty list — use `field(default_factory=list)`)
- `created_by: str` (defaults to `"system"`)

Then verify that two independently-constructed empty batches don't share the same list (the mutable-default trap doesn't bite when you use `default_factory`).

In [ ]:
from dataclasses import dataclass, field

# TODO: define Batch

b1 = Batch("alpha")
b2 = Batch("beta")
assert b1.items == []
assert b2.items == []
assert b1.items is not b2.items   # not the same list — independent instances

b1.items.append(1)
assert b1.items == [1]
assert b2.items == []             # mutating b1 did not touch b2

b3 = Batch("gamma", items=[10, 20], created_by="alice")
assert b3.items == [10, 20]
assert b3.created_by == "alice"
print("ok")


## Task 3 — Frozen dataclass as a dict key

Define a `frozen` dataclass `Coord(x: int, y: int)`. Verify:
- Attempting to set an attribute after construction raises `dataclasses.FrozenInstanceError`
- You can use `Coord` as a dict key (i.e., it's hashable)

This pattern is gold for DE: composite keys like `(date, region)` modeled as a frozen dataclass are far more self-documenting than raw tuples once the project grows.

In [ ]:
from dataclasses import dataclass, FrozenInstanceError

# TODO: define Coord as a frozen @dataclass

c = Coord(1, 2)
assert c.x == 1 and c.y == 2

try:
    c.x = 99
except FrozenInstanceError:
    pass
else:
    raise AssertionError("expected FrozenInstanceError")

# usable as dict key
sales = {Coord(0, 0): 100, Coord(1, 1): 250}
assert sales[Coord(0, 0)] == 100
assert sales[Coord(1, 1)] == 250

# usable as set element
seen = {Coord(0, 0), Coord(0, 0), Coord(1, 2)}
assert len(seen) == 2
print("ok")


## Task 4 — Methods on a dataclass

Dataclasses can have regular methods. Build `Money` (from ex_46) as a `@dataclass`:
- Fields: `amount: float`, `currency: str`
- Method `add(other) -> Money` — return a new `Money` (use `dataclasses.replace` or just `Money(...)`)
- Currency mismatch → `ValueError("currency mismatch")`
- The default `__repr__` is fine — you don't need to override it (it'll look slightly different from ex_46, that's OK)

Focus: see how the dataclass saves you `__init__` + `__repr__` while still letting you add behavior.

In [ ]:
from dataclasses import dataclass

# TODO: define Money as a @dataclass with the `add` method

m = Money(10.5, "USD")
assert m.amount == 10.5
assert m.currency == "USD"

total = Money(10.5, "USD").add(Money(2.25, "USD"))
assert isinstance(total, Money)
assert total.amount == 12.75
assert total.currency == "USD"

try:
    Money(10, "USD").add(Money(5, "BRL"))
except ValueError as e:
    assert str(e) == "currency mismatch"
else:
    raise AssertionError("expected ValueError")
print("ok")
